In [ ]:
# Data root is configurable: export SYNERGPCR_BASE=/path/to/released/tables
# (defaults to ./data). All input paths below are resolved against it.
from pathlib import Path
import os
import pandas as pd
import re
import time
import json
from tqdm import tqdm
import concurrent.futures
import google.generativeai as genai
from typing_extensions import TypedDict

BASE = Path(os.environ.get("SYNERGPCR_BASE", "./data"))


# ==========================================
# 1. API Configuration
# ==========================================
# Insert your Gemini API Key here
API_KEY = os.environ.get("GEMINI_API_KEY")
genai.configure(api_key=API_KEY)

# Define the strict output schema for the LLM
# We expect a list of indications since a single text block can mention multiple diseases.
class DiseaseEntity(TypedDict):
    disease_name: str
    umls_cui: str  # e.g., C0020538
    icd_code: str  # Approximate ICD-10 or ICD-11

class DiseaseExtraction(TypedDict):
    indications: list[DiseaseEntity]

# Initialize the model using gemini-2.5-flash
model = genai.GenerativeModel(
    'models/gemini-2.5-flash-lite',
    generation_config={
        "response_mime_type": "application/json",
        "response_schema": DiseaseExtraction,
        "temperature": 0.0 # Extremely low temp for consistent factual entity linking
    }
)

In [ ]:
# ==========================================
# 2. LLM Extraction Function
# ==========================================
def extract_diseases_llm(text, retries=7):
    """Calls the Gemini API to extract and standardize diseases into UMLS CUI."""
    time.sleep(6)
    prompt = f"""
    You are an expert clinical data curator and ontologist.
    Extract the primary therapeutic indications (diseases or medical conditions) from the following text.
    For each disease, provide the standard medical name, the corresponding UMLS CUI (Concept Unique Identifier), and an approximate ICD-10/11 code.
    If a specific ID is unknown, output 'Unknown'.
    Ignore adverse effects or mechanism descriptions; focus ONLY on the diseases the drug aims to treat.

    Text: "{text}"
    """
    wait = 5
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            
            # ✅ 이 3줄이 핵심 수정
            raw = response.text.strip()
            raw = re.sub(r"^```(?:json)?\s*", "", raw)
            raw = re.sub(r"\s*```$", "", raw)
            
            return json.loads(raw)
        except json.JSONDecodeError as je:
            # JSON 파싱 실패 시 raw 내용을 출력해서 확인 가능하게
            print(f"JSON parse error: {je}\nRaw response: {raw[:200]}")
            break
        except Exception as e:
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"Rate limit. Waiting {wait}s... (attempt {attempt+1}/{retries})")
                time.sleep(wait)
                wait *= 2
            else:
                print(f"Extraction error: {e}")
                break
    return {"indications": []}


In [ ]:
# ==========================================
# 3. Processing DrugBank
# ==========================================
def process_drugbank_indications(df_drugbank, output_dir):
    print("\n--- Processing DrugBank Indications ---")
    
    CHECKPOINT_FILE = os.path.join(output_dir, "drugbank_checkpoint.json")
    
    # Load existing checkpoint if available
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            mapping_results = json.load(f)
        print(f"Resumed from checkpoint: {len(mapping_results)} already done.")
    else:
        mapping_results = {}

    unique_texts = df_drugbank['Indication'].dropna().unique()
    print(f"Found {len(unique_texts)} unique Indication texts in DrugBank.")

    for i, text in enumerate(tqdm(unique_texts, desc='Mining DrugBank LLM')):
        # Skip already processed texts
        if text in mapping_results:
            continue

        result = extract_diseases_llm(text)
        names = [d.get('disease_name', '') for d in result.get('indications', [])]
        cuis  = [d.get('umls_cui', '') for d in result.get('indications', [])]
        icds  = [d.get('icd_code', '')  for d in result.get('indications', [])]

        mapping_results[text] = {
            'Original_Text':          text,
            'Standard_Disease_Names': '|'.join(filter(None, names)),
            'UMLS_CUI':               '|'.join(filter(None, cuis)),
            'ICD_Code':               '|'.join(filter(None, icds))
        }

        # Save checkpoint every 50 iterations
        if (i + 1) % 50 == 0:
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump(mapping_results, f, ensure_ascii=False)
            print(f"Checkpoint saved at {i+1} texts.")

    # Final save
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(mapping_results, f, ensure_ascii=False)

    mapping_df = pd.DataFrame.from_dict(mapping_results, orient='index')
    df_drugbank = df_drugbank.merge(mapping_df, left_on='Indication', right_on='Original_Text', how='left')
    df_drugbank.drop(columns=['Original_Text'], inplace=True)

    out_path = os.path.join(output_dir, 'DrugBank_Master_Standardized.csv')
    df_drugbank.to_csv(out_path, index=False)
    print(f"DrugBank standardization complete. Saved to: {out_path}")

    return df_drugbank

In [ ]:
def extract_umls_from_disease_name(disease_name, icd_code):
    """TTD용 - 짧은 disease name에서 UMLS CUI 추출"""
    time.sleep(6)
    prompt = f"""
    You are a biomedical ontology expert.
    Given a disease name and its ICD code, return ONLY the UMLS CUI for this exact disease.
    
    Disease name: "{disease_name}"
    ICD code: "{icd_code}"
    
    Rules:
    - Return the single most specific UMLS CUI for this disease
    - If truly unknown, return "Unknown"
    - Do NOT return multiple CUIs
    """
    
    # TypedDict 스키마 없이 단순 텍스트로 받기
    simple_model = genai.GenerativeModel(
        'gemini-2.5-flash-lite',
        generation_config={"temperature": 0.0}
    )
    
    wait = 5
    for attempt in range(7):
        try:
            response = simple_model.generate_content(prompt)
            cui = response.text.strip()
            # UMLS CUI 패턴 검증 (C + 7자리 숫자)
            match = re.search(r'C\d{7}', cui)
            return match.group(0) if match else 'Unknown'
        except Exception as e:
            if "429" in str(e) or "ResourceExhausted" in str(e):
                print(f"Rate limit. Waiting {wait}s... (attempt {attempt+1}/7)")
                time.sleep(wait)
                wait *= 2
            else:
                print(f"Error: {e}")
                break
    return 'Unknown'

In [ ]:
# ==========================================
# 4. Processing TTD (Target & Drug Indications)
# ==========================================
def process_ttd_indications(df_ttd, output_dir):
    print("\n--- Processing TTD Indications ---")

    CHECKPOINT_FILE = os.path.join(output_dir, "ttd_checkpoint.json")

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            mapping_results = json.load(f)
        print(f"Resumed from checkpoint: {len(mapping_results)} already done.")
    else:
        mapping_results = {}

    ttd_target_dis = df_ttd['Target_Biological_Diseases'].dropna().str.split('|').explode().unique()
    ttd_drug_dis   = df_ttd['Drug_Clinical_Indications'].dropna().str.split('|').explode().unique()
    unique_ttd_diseases = list(set(ttd_target_dis).union(set(ttd_drug_dis)))
    print(f"Found {len(unique_ttd_diseases)} unique disease entities in TTD.")

    # ✅ TTD 문자열에서 disease name과 ICD 코드를 먼저 regex로 분리
    def parse_ttd_string(disease_str):
        """
        TTD format examples:
          'Glaucoma/ocular hypertension (ICD-11: 9C61)'
          'Abortion [ICD-11: JA00]'
        Returns (clean_name, icd_code)
        """
        match = re.search(r'\(ICD-\d+:\s*([^\)]+)\)|\[ICD-\d+:\s*([^\]]+)\]', disease_str)
        if match:
            icd = (match.group(1) or match.group(2)).strip()
            clean_name = re.sub(r'\s*[\(\[]ICD-\d+:[^\)\]]+[\)\]]', '', disease_str).strip()
        else:
            icd = 'Unknown'
            clean_name = disease_str.strip()
        return clean_name, icd

    # ✅ 순차 처리 + 체크포인트
    for i, disease_str in enumerate(tqdm(unique_ttd_diseases, desc='Mining TTD LLM')):
        if disease_str in mapping_results:
            continue

        clean_name, icd_code = parse_ttd_string(disease_str)
        
        umls = extract_umls_from_disease_name(clean_name, icd_code)

        mapping_results[disease_str] = {
            'UMLS_CUI': umls,
            'ICD_Code': icd_code
        }

        if (i + 1) % 50 == 0:
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump(mapping_results, f, ensure_ascii=False)
            print(f"Checkpoint saved at {i+1} texts.")

    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(mapping_results, f, ensure_ascii=False)

    # ✅ 파이프 구분 문자열을 컬럼별로 매핑
    def map_col(piped_string, field):
        if pd.isna(piped_string):
            return None
        return '|'.join(
            mapping_results.get(item, {}).get(field, 'Unknown')
            for item in piped_string.split('|')
        )

    df_ttd['Target_Biological_Diseases_UMLS'] = df_ttd['Target_Biological_Diseases'].apply(lambda x: map_col(x, 'UMLS_CUI'))
    df_ttd['Target_Biological_Diseases_ICD']  = df_ttd['Target_Biological_Diseases'].apply(lambda x: map_col(x, 'ICD_Code'))
    df_ttd['Drug_Clinical_Indications_UMLS']  = df_ttd['Drug_Clinical_Indications'].apply(lambda x: map_col(x, 'UMLS_CUI'))
    df_ttd['Drug_Clinical_Indications_ICD']   = df_ttd['Drug_Clinical_Indications'].apply(lambda x: map_col(x, 'ICD_Code'))

    out_path = os.path.join(output_dir, 'TTD_Master_Standardized.csv')
    df_ttd.to_csv(out_path, index=False)
    print(f"TTD standardization complete. Saved to: {out_path}")
    return df_ttd

In [ ]:
# ==========================================
# 5. Execution Block
# ==========================================
# Define your paths
drugbank_dir = str(BASE / "Output/DB/DrugBank/NAR/")
ttd_dir = str(BASE / "Output/DB/TTD/NAR/")

# Load previously parsed datasets
drugbank_path = str(BASE / "Output/DB/DrugBank/NAR/DrugBank_DrugInfo_v2.csv")
ttd_path = str(BASE / "Output/DB/TTD/NAR/TTD_GPCR_Master.csv")

df_drugbank = pd.read_csv(drugbank_path)
df_ttd = pd.read_csv(ttd_path)

# Process
df_drugbank_standardized = process_drugbank_indications(df_drugbank, drugbank_dir)
df_ttd_standardized = process_ttd_indications(df_ttd, ttd_dir)

In [ ]:
df_ttd_standardized = process_ttd_indications(df_ttd, ttd_dir)

In [ ]:
ttd = pd.read_csv(str(BASE / "Output/DB/TTD/NAR/TTD_Master_Standardized.csv"))

In [ ]:
ttd

IUPHAR

In [ ]:
def process_iuphar_indications(df_iuphar, output_dir):
    print("\n--- Processing IUPHAR Indications ---")

    CHECKPOINT_FILE = os.path.join(output_dir, "iuphar_checkpoint.json")

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            mapping_results = json.load(f)
        print(f"Resumed from checkpoint: {len(mapping_results)} already done.")
    else:
        mapping_results = {}

    # InChIKey 기준으로 unique 처리 (같은 약물이 여러 Target에 매핑되므로)
    unique_texts = df_iuphar.dropna(subset=['Clinical Use'])[['InChIKey', 'Clinical Use']].drop_duplicates('InChIKey')
    print(f"Found {len(unique_texts)} unique Clinical Use texts in IUPHAR.")

    for i, (_, row) in enumerate(tqdm(unique_texts.iterrows(), total=len(unique_texts), desc='Mining IUPHAR LLM')):
        key = row['InChIKey']
        text = row['Clinical Use']

        if key in mapping_results:
            continue

        result = extract_diseases_llm(text)
        names = [d.get('disease_name', '') for d in result.get('indications', [])]
        cuis  = [d.get('umls_cui', '')      for d in result.get('indications', [])]
        icds  = [d.get('icd_code', '')      for d in result.get('indications', [])]

        mapping_results[key] = {
            'Standard_Disease_Names': '|'.join(filter(None, names)),
            'UMLS_CUI':               '|'.join(filter(None, cuis)),
            'ICD_Code':               '|'.join(filter(None, icds))
        }

        if (i + 1) % 50 == 0:
            with open(CHECKPOINT_FILE, "w") as f:
                json.dump(mapping_results, f, ensure_ascii=False)
            print(f"Checkpoint saved at {i+1} texts.")

    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(mapping_results, f, ensure_ascii=False)

    # InChIKey 기준으로 merge (1:N 관계 자동 처리)
    mapping_df = pd.DataFrame.from_dict(mapping_results, orient='index').reset_index()
    mapping_df.rename(columns={'index': 'InChIKey'}, inplace=True)

    df_iuphar = df_iuphar.merge(mapping_df, on='InChIKey', how='left')

    out_path = os.path.join(output_dir, 'IUPHAR_GPCR_Indications_Standardized.csv')
    df_iuphar.to_csv(out_path, index=False)
    print(f"IUPHAR standardization complete. Saved to: {out_path}")

    return df_iuphar

In [ ]:
iuphar_dir = str(BASE / "Output/DB/IUPHAR/NAR/")
iuphar_path = str(BASE / "Output/DB/IUPHAR/NAR/IUPHAR_GPCR_Indications_Raw.csv")

df_iuphar = pd.read_csv(iuphar_path)
df_iuphar_standardized = process_iuphar_indications(df_iuphar, iuphar_dir)

In [ ]:
iup = pd.read_csv(str(BASE / "Output/DB/IUPHAR/NAR/IUPHAR_GPCR_Indications_Standardized.csv"))

In [ ]:
iup